
# RAGAS + Langfuse — Continuous Evaluation (SDK v4)

**Day 4 — AI Security & Legal Compliance · Practical 3 of 4 · Companion to the "Evals &
Observability" deck**

> **Running in Google Colab:** works on the default **CPU runtime**. Uses **Langfuse Cloud's
> free tier** (50k observations/month, no credit card) — no self-hosted server required.

> **SDK versions:** this notebook targets **Langfuse Python SDK v4** (currently 4.14.x) and
> **RAGAS >= 0.2**. Two rewrites have happened here, so old snippets fail loudly:
>
> - **v2 → v3**: OpenTelemetry rewrite. `langfuse.trace()`, `trace.span()`, `langfuse.score()`
>   removed, and v2 trace ingestion is no longer accepted by current Langfuse Cloud — symptom:
>   scores appear in the dashboard but the Traces panel stays empty.
> - **v3 → v4**: `start_as_current_span()` / `start_as_current_generation()` collapsed into a
>   single `start_as_current_observation(..., as_type=...)`.
>
> Check what you actually have with `langfuse.__version__` before debugging anything else.

---

## Learning Objectives

By the end of this notebook, you will be able to:

1. Capture full traces of RAG calls in Langfuse
2. Score those traces automatically with RAGAS as an evaluator function
3. See the deck's "continuous eval loop" running end to end, on real (if small-scale) traffic

## Why This Matters for a Law Firm

This is the observability layer that makes "is our legal-AI tool still working well" an
answerable, measured question instead of a guess -- every trace scored, visible in one
dashboard, comparable run over run.

## Notebook Workflow

```mermaid
flowchart LR
    A["RAG query"] --> B["Langfuse trace\ncaptured"]
    B --> C["RAGAS evaluator\nfunction scores it"]
    C --> D["Score attached\nto Langfuse trace"]
    D --> E["Langfuse dashboard:\ncomparable across runs"]
```



## Section 1 — Setup

Sign up for a free Langfuse Cloud account if you haven't already, then create a project and
copy its public/secret API keys into Colab Secrets.

**Pick the region that matches your dashboard URL** — keys are region-scoped, and a mismatch
fails authentication:

| Dashboard URL | `LANGFUSE_BASE_URL` |
|---|---|
| `us.cloud.langfuse.com` | `https://us.cloud.langfuse.com` |
| `cloud.langfuse.com` | `https://cloud.langfuse.com` |


In [ ]:

%pip install -q "langfuse>=4" "ragas>=0.2" datasets openai langchain-openai

import os

# Region of your Langfuse Cloud project -- must match your dashboard URL.
# v4 reads LANGFUSE_BASE_URL; LANGFUSE_HOST still works but is deprecated.
LANGFUSE_BASE_URL = "https://us.cloud.langfuse.com"

def get_api_key(env_var_name):
    try:
        from google.colab import userdata
        key = userdata.get(env_var_name)
        if key:
            return key
    except ImportError:
        pass
    return os.environ.get(env_var_name)

LANGFUSE_PUBLIC_KEY = get_api_key("LANGFUSE_PUBLIC_KEY")
LANGFUSE_SECRET_KEY = get_api_key("LANGFUSE_SECRET_KEY")
OPENAI_API_KEY = get_api_key("OPENAI_API_KEY")

for name, value in [
    ("LANGFUSE_PUBLIC_KEY", LANGFUSE_PUBLIC_KEY),
    ("LANGFUSE_SECRET_KEY", LANGFUSE_SECRET_KEY),
    ("OPENAI_API_KEY", OPENAI_API_KEY),
]:
    if not value:
        raise ValueError(f"Add a {name} secret in Colab (key icon, left sidebar).")
    os.environ[name] = value

# SDK v4 reads its config from the environment.
os.environ["LANGFUSE_BASE_URL"] = LANGFUSE_BASE_URL

print("All API keys loaded.")



## Section 2 — Initialize Langfuse and a Minimal RAG Function

We rebuild a compact RAG pipeline (same pattern as Day 3's notebooks), and wrap each call with
Langfuse's tracing so every question/answer/context triple gets captured as a trace.

**v4 tracing model:** there is no separate `trace` object any more. A trace is simply the
outermost observation, created with `langfuse.start_as_current_observation(...)`; nested
observations are opened inside it as context managers and closed automatically on exit. The
kind of observation is chosen with `as_type=` — `"span"` for plain work, `"generation"` for an
LLM call (also `"tool"`, `"retriever"`, `"agent"`, `"embedding"`, `"guardrail"`, …), which is
what drives the icons and the cost accounting in the dashboard. The trace id comes from
`observation.trace_id`.

We instrument by hand here because seeing the spans built explicitly is the teaching point. In
production you would more likely use the drop-in `from langfuse.openai import openai`, which
auto-traces every OpenAI call (model, tokens, cost, errors) without the `with` block.

`langfuse.auth_check()` is worth keeping — it fails loudly here on a bad key or wrong region,
instead of silently dropping every trace later.


In [ ]:

from langfuse import get_client
from openai import OpenAI

langfuse = get_client()  # reads LANGFUSE_PUBLIC_KEY / LANGFUSE_SECRET_KEY / LANGFUSE_BASE_URL

if not langfuse.auth_check():
    raise RuntimeError(
        f"Langfuse auth failed against {LANGFUSE_BASE_URL}. "
        "Check the keys AND that the host matches the region your project lives in."
    )

client = OpenAI(api_key=OPENAI_API_KEY)

legal_context_db = {
    "termination": "Section 9.2: Either party may terminate this Agreement upon sixty (60) days' written notice.",
    "liability": "Section 14.3: Total liability shall not exceed fees paid in the preceding twelve (12) months.",
    "indemnification": "Section 7.1: The Contractor shall indemnify the Client for claims arising from gross negligence.",
}

def simple_retrieve(query):
    query_lower = query.lower()
    for key, text in legal_context_db.items():
        if key in query_lower:
            return [text]
    return [list(legal_context_db.values())[0]]  # fallback: return something, for demo purposes


def traced_rag_query(question):
    with langfuse.start_as_current_observation(
        name="legal-rag-query", as_type="span", input={"question": question}
    ) as root_span:

        with langfuse.start_as_current_observation(
            name="retrieval", as_type="retriever", input={"question": question}
        ) as retrieval_span:
            contexts = simple_retrieve(question)
            retrieval_span.update(output={"contexts": contexts})

        with langfuse.start_as_current_observation(
            name="generation", as_type="generation", model="gpt-4o-mini", input=contexts
        ) as generation_span:
            response = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[
                    {"role": "system", "content": "Answer using only the provided context."},
                    {"role": "user", "content": f"Context: {contexts}\n\nQuestion: {question}"},
                ],
                temperature=0,
            )
            answer = response.choices[0].message.content
            generation_span.update(
                output=answer,
                usage_details={
                    "input": response.usage.prompt_tokens,
                    "output": response.usage.completion_tokens,
                },
            )

        root_span.update(output={"answer": answer})
        trace_id = root_span.trace_id

    return answer, contexts, trace_id

print("Traced RAG function ready.")



## Section 3 — Run Some Queries and Capture Traces

Each call here creates a full trace in Langfuse -- visible in your Langfuse Cloud dashboard
under **Tracing**, with spans for retrieval and generation separately.

If the Traces panel stays empty while Scores fill up later, that is the classic v2-SDK
symptom described at the top of this notebook — re-check that `langfuse>=4` is what actually
got installed. The version print below is there for exactly that reason.


In [ ]:

import langfuse as langfuse_pkg
print("Langfuse SDK version:", langfuse_pkg.__version__)

questions = [
    "How much notice is needed to terminate?",
    "What is the liability cap?",
    "What does indemnification cover here?",
]

trace_records = []
for q in questions:
    answer, contexts, trace_id = traced_rag_query(q)
    trace_records.append({"question": q, "answer": answer, "contexts": contexts, "trace_id": trace_id})
    print(f"Q: {q}\nA: {answer}\nTrace ID: {trace_id}\n")

langfuse.flush()  # ensure all traces are sent before moving on
print(f"\n{len(trace_records)} traces captured and flushed to Langfuse Cloud.")



## Section 4 — Score the Traces with RAGAS

RAGAS's metrics are reference-free, so they can run directly against these captured
question/answer/context triples -- no hand-labeled ground truth needed, exactly the property
that makes them usable on live production traffic, not just a static test set.

**Column naming:** RAGAS >= 0.2 renamed its schema — `question` → `user_input`,
`answer` → `response`, `contexts` → `retrieved_contexts`, `ground_truth` → `reference`. Legacy
names are still accepted on input, but `to_pandas()` always returns the **new** names, which is
why indexing the result with `"question"` raises `KeyError: "['question'] not in index"`. We
use the new names on both sides.


In [ ]:

from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

eval_dataset = Dataset.from_dict({
    "user_input": [r["question"] for r in trace_records],
    "response": [r["answer"] for r in trace_records],
    "retrieved_contexts": [r["contexts"] for r in trace_records],
})

ragas_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini", temperature=0, api_key=OPENAI_API_KEY))
ragas_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(api_key=OPENAI_API_KEY))

ragas_results = evaluate(
    dataset=eval_dataset,
    metrics=[faithfulness, answer_relevancy],
    llm=ragas_llm,
    embeddings=ragas_embeddings,
)

scores_df = ragas_results.to_pandas()

# Version-agnostic: newer RAGAS emits "user_input", older emits "question".
question_col = "user_input" if "user_input" in scores_df.columns else "question"
print(scores_df[[question_col, "faithfulness", "answer_relevancy"]])



## Section 5 — Attach RAGAS Scores Back to Langfuse Traces

This is the integration itself: RAGAS scores get pushed back onto the SAME Langfuse trace they
were computed from, so anyone looking at a trace in the Langfuse dashboard sees both what
happened AND how it scored, in one place.

Since v3 the call is `langfuse.create_score(...)` (v2's `langfuse.score(...)` is gone). Note that
Langfuse accepts a score for a `trace_id` that does not exist — an orphan score. That is
exactly why a broken tracing setup shows "0 traces, N scores" rather than an error.


In [ ]:

for record, (_, row) in zip(trace_records, scores_df.iterrows()):
    langfuse.create_score(
        trace_id=record["trace_id"],
        name="faithfulness",
        value=float(row["faithfulness"]),
    )
    langfuse.create_score(
        trace_id=record["trace_id"],
        name="answer_relevancy",
        value=float(row["answer_relevancy"]),
    )

langfuse.flush()
print("RAGAS scores attached to their corresponding Langfuse traces.")
print("Open your Langfuse Cloud dashboard -> Tracing to see each trace with its scores attached.")



## Section 6 — Why This Matters: A Regression-Detection Scenario

Simulate the deck's CI use case: run the SAME question set again after a hypothetical pipeline
change, and compare scores across the two runs -- this is exactly how a team would catch a
regression before it reaches production.


In [ ]:

# Simulate a "changed" pipeline: same questions, but imagine a worse chunking strategy
# degraded the context deliberately for this demonstration.
degraded_context_db = {
    "termination": "This document contains various provisions.",  # deliberately vague/unhelpful
    "liability": "Section 14.3: Total liability shall not exceed fees paid in the preceding twelve (12) months.",
    "indemnification": "Section 7.1: The Contractor shall indemnify the Client for claims arising from gross negligence.",
}

def degraded_retrieve(query):
    query_lower = query.lower()
    for key, text in degraded_context_db.items():
        if key in query_lower:
            return [text]
    return [list(degraded_context_db.values())[0]]

degraded_answer_response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "Answer using only the provided context."},
        {"role": "user", "content": f"Context: {degraded_retrieve('How much notice is needed to terminate?')}\n\nQuestion: How much notice is needed to terminate?"},
    ],
    temperature=0,
)
print("Answer with DEGRADED context (simulating a retrieval regression):")
print(degraded_answer_response.choices[0].message.content)
print("\nCompare this against Section 3's original answer -- a RAGAS Faithfulness/Relevancy")
print("re-run on this new answer would score noticeably lower, flagging the regression")
print("automatically, before a human ever manually notices the quality drop.")



## Key Takeaways

1. **No self-hosted server was needed** -- Langfuse Cloud's free tier worked directly from
   Colab via API key, exactly per the feasibility research.
2. **RAGAS's reference-free metrics ran on live traces**, not a static hand-labeled test set --
   this is what makes continuous evaluation practical rather than a one-time exercise.
3. **Scores attach directly to traces**, so debugging a low score means looking at the SAME
   trace that has full visibility into what retrieval and generation actually did.
4. Section 6 demonstrates the actual payoff: a regression in retrieval quality shows up as a
   measurable score drop, catchable in CI before it ever reaches a real user.

### SDK migration notes (v2 -> v4)

| Langfuse v2 | Langfuse v3 | Langfuse v4 (this notebook) |
|---|---|---|
| `Langfuse(public_key=..., secret_key=..., host=...)` | `get_client()` + env vars | `get_client()` + env vars (`LANGFUSE_BASE_URL`; `LANGFUSE_HOST` deprecated) |
| `langfuse.trace(name=...)` | `start_as_current_span(name=...)` | `start_as_current_observation(name=..., as_type="span")` — outermost observation *is* the trace |
| `trace.span(name=...)` | nested `start_as_current_span(...)` | nested `start_as_current_observation(..., as_type="span"/"retriever"/"tool")` |
| `trace.generation(name=..., model=...)` | `start_as_current_generation(...)` | `start_as_current_observation(..., as_type="generation", model=...)` |
| `span.end(output=...)` | `span.update(output=...)` | `span.update(output=...)`, closed by the `with` block |
| `trace.id` | `span.trace_id` | `observation.trace_id` |
| `langfuse.score(trace_id=..., ...)` | `langfuse.create_score(...)` | `langfuse.create_score(trace_id=..., ...)` |

**Next up:** the *Fairness Metrics Mini-Lab* notebook — the AI Ethics deck's mini-lab, made
runnable code.
